In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS olist_ecommerce.gold;

In [0]:
%sql
-- Criando tabela via Spark SQL
CREATE OR REPLACE TABLE olist_ecommerce.gold.monthly_sales
USING DELTA
AS
SELECT
    EXTRACT(MONTH FROM order_purchase_timestamp) AS sales_month,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(*) AS total_items,

    SUM(price) AS total_sales,
    SUM(freight_value) AS total_freight,
    SUM(total_item_value) AS total_revenue,

    AVG(total_item_value) AS average_item_value
FROM olist_ecommerce.silver.sales_detail
GROUP BY
    EXTRACT(MONTH FROM order_purchase_timestamp)
ORDER BY
    sales_month;

------------------------

SELECT *
FROM olist_ecommerce.gold.monthly_sales
ORDER BY sales_month;

In [0]:
%sql
CREATE OR REPLACE TABLE olist_ecommerce.gold.monthly_category_sales
USING DELTA
AS
SELECT
    EXTRACT(MONTH FROM order_purchase_timestamp) AS sales_month,
    product_category_name_english AS category,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(price) AS total_sales,
    SUM(freight_value) AS total_freight,
    SUM(total_item_value) AS total_revenue

FROM olist_ecommerce.silver.sales_detail
GROUP BY
    EXTRACT(MONTH FROM order_purchase_timestamp),
    product_category_name_english;

------------------------

SELECT *
FROM olist_ecommerce.gold.monthly_category_sales
ORDER BY sales_month;

In [0]:
%sql
-- Quais categorias venderam mais
SELECT
    category,
    SUM(total_revenue) AS revenue
FROM olist_ecommerce.gold.monthly_category_sales
GROUP BY
    category
ORDER BY
    revenue DESC;

In [0]:
%sql
-- Quais cidades venderam mais
SELECT
    s.seller_id,
    s.seller_city,
    s.seller_state,
    SUM(sd.total_item_value) AS revenue
FROM olist_ecommerce.silver.sales_detail sd
    INNER JOIN olist_ecommerce.silver.sellers_clean s ON sd.seller_id = s.seller_id
GROUP BY
    s.seller_id,
    s.seller_city,
    s.seller_state
ORDER BY
    revenue DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE olist_ecommerce.gold.seller_ranking
USING DELTA
AS
WITH seller_sales AS (
    SELECT
        seller_id,
        seller_state,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(*) AS total_items,
        SUM(price) AS total_sales,
        SUM(freight_value) AS total_freight,
        SUM(total_item_value) AS total_revenue
    FROM olist_ecommerce.silver.sales_detail
    GROUP BY
        seller_id,
        seller_state
)

SELECT
    seller_id,
    seller_state,
    total_orders,
    total_items,
    total_sales,
    total_freight,
    total_revenue,

    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS global_rank,

    RANK() OVER (
        PARTITION BY seller_state
        ORDER BY total_revenue DESC
    ) AS state_rank

FROM seller_sales;

In [0]:
%sql
SELECT *
FROM olist_ecommerce.gold.seller_ranking

In [0]:
%sql

CREATE OR REPLACE TABLE olist_ecommerce.gold.seller_ranking
USING DELTA
AS

SELECT
    seller_id,
    seller_state,
    total_orders,
    total_items,
    total_sales,
    total_freight,
    total_revenue,

    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS global_rank,

    RANK() OVER (
        PARTITION BY seller_state
        ORDER BY total_revenue DESC
    ) AS state_rank

FROM olist_ecommerce.gold.seller_sales;

In [0]:
%sql

SELECT *
FROM olist_ecommerce.gold.seller_ranking
ORDER BY global_rank
LIMIT 20;

In [0]:
%sql

CREATE OR REPLACE TABLE olist_ecommerce.gold.monthly_category_sales
USING DELTA
AS

SELECT
    EXTRACT(MONTH FROM order_purchase_timestamp) AS sales_month,
    product_category_name_english AS category,
    SUM(total_item_value) AS total_revenue
FROM olist_ecommerce.silver.sales_detail

WHERE product_category_name_english IS NOT NULL

GROUP BY
    EXTRACT(MONTH FROM order_purchase_timestamp),
    product_category_name_english;

In [0]:
%sql

CREATE OR REPLACE TABLE olist_ecommerce.gold.category_month_pivot
USING DELTA
AS

SELECT *
FROM olist_ecommerce.gold.monthly_category_sales

PIVOT (
    SUM(total_revenue)
    FOR category IN (
        'health_beauty',
        'bed_bath_table',
        'sports_leisure',
        'computers_accessories',
        'furniture_decor',
        'housewares'
    )
)

ORDER BY sales_month;

In [0]:
%sql

SELECT *
FROM olist_ecommerce.gold.category_month_pivot
ORDER BY sales_month;